# 01 - Data Exploration

**Status: reconstruction on synthetic data.** The thesis describes training the
predictive AI model on a per-patient database of `(timestamp, day_of_week,
unit_location, applied_load_kg)` readings, but that raw database was not exported
from the original work. This notebook generates a synthetic dataset with the same
shape and the same reported summary statistics (peak approx 4 kg, average approx 0.80 kg
over a 14-minute/85-interval window - thesis Figure 3.3.3) so the rest of the
pipeline (02-06) can be run end-to-end and inspected.

See `ai/README.md` for exactly what is reconstruction vs. direct thesis output.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
pd.set_option("display.max_columns", 20)


## Generate a synthetic per-patient load database

Three location units, matching the thesis's examples: `1` = Home, `2` = Work,
`3` = Supermarket. We simulate 60 days of 10-second-interval readings, with
two injected repeated-routine patterns (see notebooks 03/04).

In [ ]:
def generate_synthetic_load_log(n_days=60, seed=42):
    rng = np.random.default_rng(seed)
    rows = []
    start = pd.Timestamp("2026-03-01")
    for day in range(n_days):
        date = start + pd.Timedelta(days=day)
        dow = date.day_name()

        # Baseline light household activity (Home, unit 1)
        n_baseline = rng.integers(40, 80)
        times = pd.to_datetime(date) + pd.to_timedelta(
            rng.integers(6*3600, 22*3600, n_baseline), unit="s"
        )
        loads = np.clip(rng.normal(1.5, 0.8, n_baseline), 0, None)
        for t, l in zip(times, loads):
            rows.append((t, dow, 1, round(l, 2)))

        # Repeated medium-load routine ~85 intervals, most days, Home (unit 1),
        # matching thesis Figure 3.3.2/3.3.3: peak ~4kg, avg ~0.80kg
        if rng.random() < 0.85:
            window_start = pd.Timestamp(date) + pd.Timedelta(hours=18, minutes=int(rng.integers(0, 30)))
            n_intervals = 85
            t_index = window_start + pd.to_timedelta(np.arange(n_intervals) * 10, unit="s")
            base = np.abs(rng.normal(0.8, 1.1, n_intervals))
            spikes = rng.random(n_intervals) < 0.34  # ~29/85 crossing normal limit
            base[spikes] += rng.uniform(2.5, 3.2, spikes.sum())
            base = np.clip(base, 0, 4.2)
            for t, l in zip(t_index, base):
                rows.append((t, dow, 1, round(float(l), 2)))

        # Repeated short high-load spike, Supermarket (unit 3), ~once every few days
        if rng.random() < 0.4:
            t = pd.Timestamp(date) + pd.Timedelta(hours=int(rng.integers(9, 19)))
            n_spike = 10
            spike_times = t + pd.to_timedelta(np.arange(n_spike) * 5, unit="s")
            spike_loads = np.clip(rng.normal(8.5, 0.6, n_spike), 5, 12)
            for st, sl in zip(spike_times, spike_loads):
                rows.append((st, dow, 3, round(float(sl), 2)))

    df = pd.DataFrame(rows, columns=["timestamp", "day_of_week", "unit_location", "applied_load_kg"])
    return df.sort_values("timestamp").reset_index(drop=True)

df = generate_synthetic_load_log()
df.head(10)


In [ ]:
print("Rows:", len(df))
print("Date range:", df["timestamp"].min(), "to", df["timestamp"].max())
print("Missing values:")
print(df.isna().sum())
df["applied_load_kg"].describe()


## Distribution and time patterns

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
df["applied_load_kg"].hist(bins=40, ax=axes[0])
axes[0].set_title("Load distribution (kg)")
axes[0].set_xlabel("Applied load (kg)")

df.groupby("unit_location")["applied_load_kg"].mean().plot(kind="bar", ax=axes[1])
axes[1].set_title("Average load by location unit")
axes[1].set_xlabel("Unit location (1=Home, 2=Work, 3=Supermarket)")
plt.tight_layout()
plt.savefig("../results/exploration_distribution.png", dpi=120)
plt.show()


In [ ]:
df.to_csv("../data/synthetic_load_log.csv", index=False)
print("Saved ../data/synthetic_load_log.csv")
